# KeelNet Final Comparison Template

Use this notebook after the method stages are done to compare the strongest completed artifacts in one place.

Recommended flow:

1. run setup and validation
2. confirm the discovered eval paths
3. generate the comparison table and figures
4. save the summary artifacts for the paper


## Comparison Notes

Use this notebook after meaningful stage runs already exist in the artifact store.

### Goal

Build one trustworthy comparison across the strongest completed runs without turning this into another method stage.

### What This Notebook Should Produce

- one comparison table across Stage 1 baseline, Stage 1 abstain, Stage 4, Stage 5, Stage 6, Stage 7, and Stage 8 when artifacts exist
- paper-ready figures saved into the run folder
- a short written takeaway that says which trade-off looks strongest and which stages are still missing

### Interpretation Rules

- do not overclaim from missing rows
- prefer trade-off plots over single-metric winners
- treat unsupported-answer rate as a primary constraint, not a side metric


<h2 style="color: #1d4ed8;">1. Setup And Sync</h2>

Run this cell in either hosted Google Colab or Google Colab connected to a local Jupyter runtime.

What it does:

- hosted Colab: mounts Drive, loads `HF_TOKEN` from Colab Secrets if available, clones or updates `/content/KeelNet`, checks out the stage branch, and installs the project
- local runtime: reuses your local repo, uses a local project folder for artifacts, reads `HF_TOKEN` from the environment if available, and installs the project into the current kernel environment

Path reminder:

- hosted Colab defaults: repo `/content/KeelNet`, project folder `/content/drive/MyDrive/KeelNet`
- local runtime defaults: repo `/content/KeelNet` if present, otherwise your current local checkout; project folder `/content/KeelNet-local`
- optional overrides: `KEELNET_REPO_DIR`, `KEELNET_PROJECT_DIR`, and `KEELNET_DRIVE_SYNC_DIR`

Reliability reminder:

- hosted Colab defaults to a fresh clone on each setup run to avoid stale or conflicted repo state
- set `FORCE_FRESH_CLONE = False` in the setup cell or `KEELNET_FORCE_FRESH_CLONE=0` if you intentionally want to reuse `/content/KeelNet`


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys


GIT_REPO_URL = "https://github.com/naufalkmd/KeelNet.git"
DEFAULT_GIT_BRANCH = 'main'
GIT_BRANCH = os.environ.get("KEELNET_GIT_BRANCH", DEFAULT_GIT_BRANCH)
HOSTED_COLAB_PROJECT_DIR = Path("/content/drive/MyDrive/KeelNet")
DEFAULT_LOCAL_PROJECT_DIR = Path("/content/KeelNet-local")
DEFAULT_LOCAL_REPO_DIR = Path("/content/KeelNet")
FORCE_FRESH_CLONE = True

env_force_fresh_clone = os.environ.get("KEELNET_FORCE_FRESH_CLONE")
if env_force_fresh_clone is not None:
    FORCE_FRESH_CLONE = env_force_fresh_clone.strip().lower() not in ('0', 'false', 'no')


def detect_runtime_mode() -> str:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return "local-runtime"
    return "hosted-colab"


RUNTIME_MODE = detect_runtime_mode()
IS_HOSTED_COLAB = RUNTIME_MODE == "hosted-colab"
PROJECT_STORAGE_LABEL = "Drive project dir" if IS_HOSTED_COLAB else "Local project dir"


def run_setup(cmd, *, cwd: Path | None = None) -> None:
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    subprocess.run(rendered, check=True, cwd=str(cwd) if cwd else None)


def configure_project_storage() -> Path:
    if IS_HOSTED_COLAB:
        from google.colab import drive

        project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(HOSTED_COLAB_PROJECT_DIR)))
        drive.mount("/content/drive", force_remount=False)
        if not str(project_dir).startswith("/content/drive/"):
            raise ValueError(
                f"KEELNET_PROJECT_DIR must point inside /content/drive in hosted Colab, got: {project_dir}"
            )
        project_dir.mkdir(parents=True, exist_ok=True)
        print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
        return project_dir

    project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(DEFAULT_LOCAL_PROJECT_DIR))).expanduser().resolve()
    project_dir.mkdir(parents=True, exist_ok=True)
    print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
    return project_dir


def configure_drive_project_dir(project_storage_dir: Path) -> Path | None:
    if IS_HOSTED_COLAB:
        print(f"Drive project dir: {project_storage_dir}")
        return project_storage_dir.resolve()

    env_drive_dir = os.environ.get("KEELNET_DRIVE_SYNC_DIR")
    if not env_drive_dir:
        print(
            "Drive project dir: disabled "
            "(set KEELNET_DRIVE_SYNC_DIR to a local Google Drive sync folder to mirror artifacts there)."
        )
        return None

    drive_project_dir = Path(env_drive_dir).expanduser().resolve()
    drive_project_dir.mkdir(parents=True, exist_ok=True)
    print(f"Drive project dir: {drive_project_dir}")
    return drive_project_dir


def configure_hf_token() -> None:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in the environment.")
        return

    if not IS_HOSTED_COLAB:
        print("HF_TOKEN not set in the environment; continuing with anonymous HF access.")
        return

    try:
        from google.colab import userdata
    except ImportError:
        print("Colab secrets are unavailable; continuing without HF_TOKEN.")
        return

    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        os.environ["HF_TOKEN"] = token
        print("Loaded HF_TOKEN from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.")


def resolve_local_repo_dir() -> Path:
    candidates: list[Path] = []
    env_repo_dir = os.environ.get("KEELNET_REPO_DIR")
    if env_repo_dir:
        candidates.append(Path(env_repo_dir).expanduser())
    candidates.append(DEFAULT_LOCAL_REPO_DIR)
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / ".git").exists() and (resolved / "pyproject.toml").exists():
            return resolved

    raise FileNotFoundError(
        "Could not find the KeelNet repo. Set KEELNET_REPO_DIR to your local checkout before running this cell."
    )


def ensure_repo() -> Path:
    if not IS_HOSTED_COLAB:
        return resolve_local_repo_dir()

    local_repo_dir = Path(os.environ.get("KEELNET_REPO_DIR", str(DEFAULT_LOCAL_REPO_DIR)))
    if FORCE_FRESH_CLONE and local_repo_dir.exists():
        os.chdir("/content")
        print(f"Removing existing hosted Colab repo for a fresh clone: {local_repo_dir}")
        shutil.rmtree(local_repo_dir)

    if (local_repo_dir / ".git").exists():
        run_setup(["git", "fetch", "origin"], cwd=local_repo_dir)
        run_setup(["git", "checkout", GIT_BRANCH], cwd=local_repo_dir)
        run_setup(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=local_repo_dir)
    else:
        run_setup(["git", "clone", "--branch", GIT_BRANCH, GIT_REPO_URL, str(local_repo_dir)])
    return local_repo_dir.resolve()


PROJECT_STORAGE_DIR = configure_project_storage()
DRIVE_PROJECT_DIR = configure_drive_project_dir(PROJECT_STORAGE_DIR)
configure_hf_token()
REPO_DIR = ensure_repo().resolve()
os.chdir(REPO_DIR)
print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Runtime repo dir: {REPO_DIR}")
print(f"Force fresh clone: {FORCE_FRESH_CLONE}")
CURRENT_BRANCH = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Git branch: {CURRENT_BRANCH}")
run_setup([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


def mirror_output_root(output_root: Path) -> Path | None:
    if DRIVE_PROJECT_DIR is None:
        print("Drive artifact mirror is disabled for this runtime.")
        return None

    output_root = Path(output_root).expanduser().resolve()
    if not output_root.exists():
        print(f"Nothing to mirror yet: {output_root}")
        return None

    drive_project_dir = DRIVE_PROJECT_DIR.expanduser().resolve()
    try:
        relative_output = output_root.relative_to(PROJECT_STORAGE_DIR.expanduser().resolve())
    except ValueError:
        relative_output = Path("artifacts") / output_root.name
    drive_output_root = drive_project_dir / relative_output

    if output_root == drive_output_root:
        print(f"Artifacts already stored in Drive: {output_root}")
        return output_root

    drive_output_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(output_root, drive_output_root, dirs_exist_ok=True)
    print(f"Mirrored artifacts to Drive: {drive_output_root}")
    return drive_output_root


<h2 style="color: #1d4ed8;">2. Configure The Comparison Run</h2>

Set `AUTHOR_NAME` to your name if you want the notebook to prefer your own completed runs first.

This notebook auto-finds the latest completed artifact for:

- Stage 1 baseline and abstain
- Stage 4 fixed control
- Stage 5 support-constrained learner
- Stage 6 adaptive balancer
- Stage 7 risk-budgeted action learner
- Stage 8 hybrid Stage 5 plus calibrated control

You can override any discovered path directly in the config cell before building the comparison.


In [ ]:
from pathlib import Path
import json
import re
import subprocess
import sys

import torch

REPO_DIR = Path(REPO_DIR).resolve()
PROJECT_STORAGE_DIR = Path(PROJECT_STORAGE_DIR).resolve()
DRIVE_PROJECT_DIR = Path(DRIVE_PROJECT_DIR).resolve() if DRIVE_PROJECT_DIR is not None else None

AUTHOR_NAME = "naufal"
RUN_BASENAME = f"{AUTHOR_NAME}-final-comparison"
ARTIFACTS_ROOT = PROJECT_STORAGE_DIR / "artifacts" / "final_comparison_colab"
COMPLETION_MARKER_NAME = "RUN_COMPLETED.txt"

NOTEBOOK_TITLE = "KeelNet Final Comparison"
NOTEBOOK_OBJECTIVE = "Aggregate the strongest completed stage artifacts into one comparison table and one reusable figure set."
TARGET_METRICS = [
    "overall F1",
    "answerable F1",
    "unsupported-answer rate",
    "abstain F1",
    "answer rate",
    "supported-answer rate",
]
SUGGESTED_FIGURES = [
    "overall F1 vs unsupported-answer rate",
    "answerable F1 vs unsupported-answer rate",
    "summary bar chart across key metrics",
    "answer rate vs supported-answer rate for support-aware stages",
]


def completed_versions(root: Path, run_basename: str) -> list[int]:
    versions: list[int] = []
    if not root.exists():
        return versions

    pattern = re.compile(rf"^{re.escape(run_basename)}-v(\d+)$")
    for child in root.iterdir():
        if not child.is_dir():
            continue
        match = pattern.match(child.name)
        if match and (child / COMPLETION_MARKER_NAME).exists():
            versions.append(int(match.group(1)))
    return sorted(versions)


def completed_run_dirs(root: Path, *, run_prefix: str | None = None) -> list[Path]:
    if not root.exists():
        return []

    pattern = re.compile(rf"^{re.escape(run_prefix)}-v(\d+)$") if run_prefix is not None else None
    runs: list[Path] = []
    for child in root.iterdir():
        if not child.is_dir():
            continue
        if not (child / COMPLETION_MARKER_NAME).exists():
            continue
        if pattern is not None and not pattern.match(child.name):
            continue
        runs.append(child)
    return sorted(runs, key=lambda path: (path.stat().st_mtime, path.name))


def default_upstream_path(stage_folder: str, relative_path: str, *, preferred_run_prefix: str | None = None) -> str | None:
    stage_root = PROJECT_STORAGE_DIR / "artifacts" / stage_folder
    ordered_runs: list[Path] = []

    if preferred_run_prefix is not None:
        ordered_runs.extend(reversed(completed_run_dirs(stage_root, run_prefix=preferred_run_prefix)))

    for run_dir in reversed(completed_run_dirs(stage_root)):
        if run_dir not in ordered_runs:
            ordered_runs.append(run_dir)

    relative = Path(relative_path)
    for run_dir in ordered_runs:
        candidate = run_dir / relative
        if candidate.exists():
            return str(candidate)
    return None


def normalize_optional_path(value: str | Path | None) -> Path | None:
    if value is None:
        return None
    candidate = Path(value).expanduser().resolve()
    if not candidate.exists():
        print(f"Missing artifact path: {candidate}")
        return None
    return candidate


RUN_VERSION = max(completed_versions(ARTIFACTS_ROOT, RUN_BASENAME), default=0) + 1
RUN_NAME = f"{RUN_BASENAME}-v{RUN_VERSION}"
OUTPUT_ROOT = ARTIFACTS_ROOT / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MARKER_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
RUN_NOTES_FILE = OUTPUT_ROOT / "run-notes-template.md"
RUN_SUMMARY_FILE = OUTPUT_ROOT / "run-summary.json"
COMPLETION_MARKER_FILE = OUTPUT_ROOT / COMPLETION_MARKER_NAME
FIGURES_DIR = OUTPUT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_TABLE_CSV = OUTPUT_ROOT / "comparison_metrics.csv"
COMPARISON_TABLE_JSON = OUTPUT_ROOT / "comparison_metrics.json"
COMPARISON_SUMMARY_JSON = OUTPUT_ROOT / "comparison_summary.json"

NOTEBOOK_ARCHIVE_DIR = OUTPUT_ROOT / "executed-notebook"
NOTEBOOK_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
EXECUTED_NOTEBOOK_PATH = NOTEBOOK_ARCHIVE_DIR / f"{RUN_NAME}-executed.ipynb"
EXECUTED_NOTEBOOK_INSTRUCTIONS_FILE = NOTEBOOK_ARCHIVE_DIR / "README-save-executed-notebook.txt"
EXECUTED_NOTEBOOK_INSTRUCTIONS_FILE.write_text(
    "\n".join(
        [
            "Keep the canonical stage notebook in the repo as a clean template.",
            "",
            "The save/share cells try to archive the live notebook automatically when Colab exposes it.",
            "",
            "If automatic capture is unavailable, preserve a meaningful run with outputs manually:",
            "1. in Colab, use File -> Download .ipynb or File -> Save a copy in Drive",
            f"2. save the exported file as: {EXECUTED_NOTEBOOK_PATH.name}",
            f"3. place that file inside: {NOTEBOOK_ARCHIVE_DIR}",
            "",
            "Anything saved in this run folder stays outside the template-sync path and will mirror to Drive when KEELNET_DRIVE_SYNC_DIR is enabled.",
        ]
    )
    + "\n",
    encoding="utf-8",
)
AUTO_SAVED_EXECUTED_NOTEBOOK_PATH = None


def save_executed_notebook_snapshot() -> Path | None:
    global AUTO_SAVED_EXECUTED_NOTEBOOK_PATH

    try:
        from google.colab import _message
    except ImportError:
        print(
            "Automatic executed-notebook capture is unavailable in this runtime. "
            f"If you want the notebook archive, save it manually to {EXECUTED_NOTEBOOK_PATH}."
        )
        return None

    try:
        response = _message.blocking_request("get_ipynb", timeout_sec=30)
    except TypeError:
        response = _message.blocking_request("get_ipynb", request="", timeout_sec=30)
    except Exception as exc:
        print(
            "Automatic executed-notebook capture failed. "
            f"Save the notebook manually to {EXECUTED_NOTEBOOK_PATH}. Error: {exc}"
        )
        return None

    notebook = response.get("ipynb") if isinstance(response, dict) else None
    if notebook is None:
        print(
            "Automatic executed-notebook capture did not return notebook content. "
            f"Save the notebook manually to {EXECUTED_NOTEBOOK_PATH}."
        )
        return None

    metadata = notebook.setdefault("metadata", {})
    keelnet_metadata = metadata.setdefault("keelnet", {})
    keelnet_metadata["run_name"] = RUN_NAME
    keelnet_metadata["executed_notebook_target"] = str(EXECUTED_NOTEBOOK_PATH)
    source_notebook_name = response.get("path") if isinstance(response, dict) else None
    if source_notebook_name:
        keelnet_metadata["source_notebook_name"] = source_notebook_name

    EXECUTED_NOTEBOOK_PATH.write_text(
        json.dumps(notebook, indent=1, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    AUTO_SAVED_EXECUTED_NOTEBOOK_PATH = EXECUTED_NOTEBOOK_PATH
    print(f"Saved executed notebook snapshot: {EXECUTED_NOTEBOOK_PATH}")
    return EXECUTED_NOTEBOOK_PATH

STAGE1_BASELINE_EVAL_PATH = normalize_optional_path(
    default_upstream_path("stage1_colab", "baseline_eval.json", preferred_run_prefix=f"{AUTHOR_NAME}-stage1")
)
STAGE1_ABSTAIN_EVAL_PATH = normalize_optional_path(
    default_upstream_path("stage1_colab", "abstain_eval.json", preferred_run_prefix=f"{AUTHOR_NAME}-stage1")
)
STAGE4_CONTROL_EVAL_PATH = normalize_optional_path(
    default_upstream_path("stage4_colab", "control_eval.json", preferred_run_prefix=f"{AUTHOR_NAME}-stage4")
)
STAGE5_LEARNER_EVAL_PATH = normalize_optional_path(
    default_upstream_path("stage5_colab", "learner_eval.json", preferred_run_prefix=f"{AUTHOR_NAME}-stage5")
)
STAGE6_BALANCE_EVAL_PATH = normalize_optional_path(
    default_upstream_path("stage6_colab", "balance_eval.json", preferred_run_prefix=f"{AUTHOR_NAME}-stage6")
)
STAGE7_ACTION_EVAL_PATH = normalize_optional_path(
    default_upstream_path("stage7_colab", "risk_action_eval.json", preferred_run_prefix=f"{AUTHOR_NAME}-stage7")
)
STAGE8_HYBRID_EVAL_PATH = normalize_optional_path(
    default_upstream_path("stage8_colab", "hybrid_eval.json", preferred_run_prefix=f"{AUTHOR_NAME}-stage8")
)

COMPARISON_STAGE_SPECS = [
    {
        "label": "Stage 1 Baseline",
        "family": "stage1",
        "eval_path": STAGE1_BASELINE_EVAL_PATH,
    },
    {
        "label": "Stage 1 Abstain",
        "family": "stage1",
        "eval_path": STAGE1_ABSTAIN_EVAL_PATH,
    },
    {
        "label": "Stage 4 Fixed Control",
        "family": "stage4",
        "eval_path": STAGE4_CONTROL_EVAL_PATH,
    },
    {
        "label": "Stage 5 Learner",
        "family": "stage5",
        "eval_path": STAGE5_LEARNER_EVAL_PATH,
    },
    {
        "label": "Stage 6 Adaptive Balance",
        "family": "stage6",
        "eval_path": STAGE6_BALANCE_EVAL_PATH,
    },
    {
        "label": "Stage 7 Action Learner",
        "family": "stage7",
        "eval_path": STAGE7_ACTION_EVAL_PATH,
    },
    {
        "label": "Stage 8 Hybrid Control",
        "family": "stage8",
        "eval_path": STAGE8_HYBRID_EVAL_PATH,
    },
]

AVAILABLE_STAGE_LABELS = [
    spec["label"]
    for spec in COMPARISON_STAGE_SPECS
    if spec["eval_path"] is not None
]
MISSING_STAGE_LABELS = [
    spec["label"]
    for spec in COMPARISON_STAGE_SPECS
    if spec["eval_path"] is None
]

RUN_MARKER_FILE.write_text(
    "\n".join(
        [
            f"notebook={NOTEBOOK_TITLE}",
            f"run_name={RUN_NAME}",
            f"run_version=v{RUN_VERSION}",
            f"runtime_mode={RUNTIME_MODE}",
            f"repo_dir={REPO_DIR}",
            f"project_storage_dir={PROJECT_STORAGE_DIR}",
            f"git_branch={CURRENT_BRANCH}",
            f"stage1_baseline_eval_path={STAGE1_BASELINE_EVAL_PATH}",
            f"stage1_abstain_eval_path={STAGE1_ABSTAIN_EVAL_PATH}",
            f"stage4_control_eval_path={STAGE4_CONTROL_EVAL_PATH}",
            f"stage5_learner_eval_path={STAGE5_LEARNER_EVAL_PATH}",
            f"stage6_balance_eval_path={STAGE6_BALANCE_EVAL_PATH}",
            f"stage7_action_eval_path={STAGE7_ACTION_EVAL_PATH}",
            f"stage8_hybrid_eval_path={STAGE8_HYBRID_EVAL_PATH}",
            "status=configured",
        ]
    )
    + "\n",
    encoding="utf-8",
)

print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Repo dir: {REPO_DIR}")
print(f"{PROJECT_STORAGE_LABEL}: {PROJECT_STORAGE_DIR}")
print(f"Drive project dir: {DRIVE_PROJECT_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Run basename: {RUN_BASENAME}")
print(f"Run version: v{RUN_VERSION}")
print(f"Run output dir: {OUTPUT_ROOT}")
print(f"Figures dir: {FIGURES_DIR}")
print(f"Executed notebook target: {EXECUTED_NOTEBOOK_PATH}")
print(f"Run marker file: {RUN_MARKER_FILE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Target metrics:", ", ".join(TARGET_METRICS))
print("Suggested figures:", ", ".join(SUGGESTED_FIGURES))
print("")
print("Discovered eval paths:")
for spec in COMPARISON_STAGE_SPECS:
    print(f"- {spec['label']}: {spec['eval_path']}")
print("")
print("Available rows:", ", ".join(AVAILABLE_STAGE_LABELS) if AVAILABLE_STAGE_LABELS else "none")
print("Missing rows:", ", ".join(MISSING_STAGE_LABELS) if MISSING_STAGE_LABELS else "none")


def run(cmd):
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    with subprocess.Popen(
        rendered,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    ) as process:
        if process.stdout is not None:
            for line in process.stdout:
                print(line, end="", flush=True)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, rendered)


<h2 style="color: #1d4ed8;">3. Validate The Environment</h2>

Run the project tests before stage-specific work. This confirms the installed code path is at least minimally healthy inside the current runtime.


In [ ]:
from pathlib import Path
import re


CONFLICT_MARKER_PATTERN = re.compile(r"^(<<<<<<<|=======|>>>>>>>)", re.MULTILINE)


def _problem_excerpt(path: Path, *, marker_line: int, context_lines: int = 2) -> str:
    lines = path.read_text(encoding="utf-8").splitlines()
    start = max(1, marker_line - context_lines)
    end = min(len(lines), marker_line + context_lines)
    excerpt_lines = ["--- Start of problematic section ---"]
    for line_number in range(start, end + 1):
        excerpt_lines.append(f"{line_number}: {lines[line_number - 1]}")
    excerpt_lines.append("--- End of problematic section ---")
    return "\n".join(excerpt_lines)


def assert_repo_has_no_conflict_markers(repo_dir: Path) -> None:
    candidate_paths: list[Path] = []
    for relative_path in ("pyproject.toml",):
        candidate = repo_dir / relative_path
        if candidate.exists():
            candidate_paths.append(candidate)

    for subdir in ("src", "tests"):
        root = repo_dir / subdir
        if root.exists():
            candidate_paths.extend(sorted(root.rglob("*.py")))

    for path in candidate_paths:
        try:
            text = path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            continue

        match = CONFLICT_MARKER_PATTERN.search(text)
        if match is None:
            continue

        marker_line = text.count("\n", 0, match.start()) + 1
        print(
            f"ERROR: Found potential Git merge conflict markers in {path} around line {marker_line}:"
        )
        print(_problem_excerpt(path, marker_line=marker_line))
        raise RuntimeError(
            "Please resolve these Git merge conflicts in the file directly, "
            "or rerun the setup cell to refresh /content/KeelNet, then rerun this cell."
        )


assert_repo_has_no_conflict_markers(REPO_DIR)
run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests")])


<h2 style="color: #1d4ed8;">4. Discover And Build The Comparison</h2>

Run this section after you review the printed artifact paths. It loads each available eval file, normalizes the metrics into one table, and saves comparison figures into the current run folder.


In [ ]:
print("Comparison build starts in the next cell.")


<div style="border-left: 6px solid #c2410c; background: #fff7ed; padding: 12px 16px; border-radius: 8px;">
<strong>Comparison Starts Here</strong><br/>
Sections 1-4 prepare the runtime and artifact paths. This section turns the completed stage artifacts into one comparison table and graph set.
<ul>
  <li><strong>Start here:</strong> confirm the auto-filled eval paths, then run the comparison cell below.</li>
  <li><strong>Finish here:</strong> you have a saved metrics table, saved figures, and a short textual takeaway under the comparison run folder.</li>
  <li><strong>Out of scope:</strong> retraining models, silently changing earlier metrics, or inventing missing stage results.</li>
</ul>
</div>


## IMPLEMENTATION 1: Load Stage Evals And Generate Comparison Figures

This is the main comparison section. It does four things:

1. load each available stage eval JSON
2. normalize the metrics into one table
3. save comparison figures into `FIGURES_DIR`
4. save a machine-readable comparison summary into `OUTPUT_ROOT`

Missing artifact paths are handled gracefully and reported in the notebook output.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass


def load_eval_payload(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def _stage1_answer_rate(payload: dict, metrics: dict) -> float | None:
    predictions = payload.get("dev_predictions")
    if isinstance(predictions, dict) and predictions:
        total = len(predictions)
        answered = sum(
            1
            for prediction in predictions.values()
            if str(prediction.get("decision", "abstain")).lower() == "answer"
        )
        return 100.0 * answered / total if total > 0 else None

    total = float(metrics.get("answerable_count", 0.0)) + float(metrics.get("unanswerable_count", 0.0))
    return None if total <= 0 else 100.0 - float(metrics.get("abstain_recall", 0.0))


def extract_comparison_row(spec: dict, payload: dict) -> dict:
    family = spec["family"]

    if family == "stage1":
        metrics = payload.get("dev_metrics", {}) or {}
        mix = {
            "answer_rate": _stage1_answer_rate(payload, metrics),
            "supported_answer_rate": None,
            "unsupported_among_answers_rate": None,
        }
        selected_operating_point = payload.get("selected_threshold")
    elif family == "stage4":
        metrics = payload.get("control_dev_metrics", {}) or {}
        mix = payload.get("control_dev_mix", {}) or {}
        selected_operating_point = payload.get("selected_config")
    elif family == "stage5":
        metrics = payload.get("dev_metrics", {}) or {}
        mix = payload.get("dev_mix", {}) or {}
        selected_operating_point = payload.get("selected_keep_threshold")
    elif family == "stage6":
        metrics = payload.get("dev_metrics", {}) or {}
        mix = payload.get("dev_mix", {}) or {}
        selected_operating_point = payload.get("selected_threshold")
    elif family == "stage7":
        metrics = payload.get("dev_metrics", {}) or {}
        mix = payload.get("dev_mix", {}) or {}
        overabstain = payload.get("dev_overabstain", {}) or {}
        selected_operating_point = payload.get("selected_risk_threshold")
    elif family == "stage8":
        metrics = payload.get("dev_metrics", {}) or {}
        mix = payload.get("dev_mix", {}) or {}
        selected_operating_point = payload.get("selected_candidate_threshold")
    else:
        raise ValueError(f"Unsupported comparison family: {family}")

    row = {
        "label": spec["label"],
        "family": family,
        "source_path": str(spec["eval_path"]),
        "selected_operating_point": selected_operating_point,
        "overall_f1": metrics.get("overall_f1"),
        "answerable_f1": metrics.get("answerable_f1"),
        "unsupported_answer_rate": metrics.get("unsupported_answer_rate"),
        "abstain_f1": metrics.get("abstain_f1"),
        "answer_rate": mix.get("answer_rate"),
        "supported_answer_rate": mix.get("supported_answer_rate"),
        "unsupported_among_answers_rate": mix.get("unsupported_among_answers_rate"),
        "max_unsupported_answer_rate": payload.get("max_unsupported_answer_rate"),
    }
    if family == "stage7":
        row["overabstain_rate"] = overabstain.get("overabstain_rate")
        row["max_overabstain_rate"] = payload.get("max_overabstain_rate")
    else:
        row["overabstain_rate"] = None
        row["max_overabstain_rate"] = None
    return row


comparison_records = []
missing_specs = []
for spec in COMPARISON_STAGE_SPECS:
    eval_path = spec["eval_path"]
    if eval_path is None:
        missing_specs.append(spec)
        continue
    payload = load_eval_payload(eval_path)
    comparison_records.append(extract_comparison_row(spec, payload))

comparison_df = pd.DataFrame(comparison_records)
if comparison_df.empty:
    raise RuntimeError(
        "No comparison rows were loaded. Confirm that at least one completed stage eval file exists."
    )

stage_order = [spec["label"] for spec in COMPARISON_STAGE_SPECS]
comparison_df["label"] = pd.Categorical(comparison_df["label"], categories=stage_order, ordered=True)
comparison_df = comparison_df.sort_values("label").reset_index(drop=True)

rounded_columns = [
    "overall_f1",
    "answerable_f1",
    "unsupported_answer_rate",
    "abstain_f1",
    "answer_rate",
    "supported_answer_rate",
    "unsupported_among_answers_rate",
    "overabstain_rate",
]
display_df = comparison_df.copy()
for column in rounded_columns:
    if column in display_df:
        display_df[column] = display_df[column].map(lambda value: round(float(value), 2) if pd.notna(value) else value)

print("Loaded comparison rows:")
print(", ".join(str(label) for label in comparison_df["label"].tolist()))
if missing_specs:
    print("Missing rows:")
    for spec in missing_specs:
        print(f"- {spec['label']}")
display(display_df)

COMPARISON_TABLE_CSV.write_text(display_df.to_csv(index=False), encoding="utf-8")
COMPARISON_TABLE_JSON.write_text(json.dumps(comparison_records, indent=2) + "\n", encoding="utf-8")

colors = {
    "stage1": "#475569",
    "stage4": "#0284c7",
    "stage5": "#dc2626",
    "stage6": "#7c3aed",
    "stage7": "#059669",
    "stage8": "#ea580c",
}


def _annotate_points(ax, frame: pd.DataFrame, *, x: str, y: str) -> None:
    for _, row in frame.iterrows():
        if pd.isna(row[x]) or pd.isna(row[y]):
            continue
        ax.annotate(
            str(row["label"]),
            (row[x], row[y]),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=9,
        )


overall_vs_unsupported_path = FIGURES_DIR / "overall-f1-vs-unsupported-answer-rate.png"
answerable_vs_unsupported_path = FIGURES_DIR / "answerable-f1-vs-unsupported-answer-rate.png"
summary_bar_path = FIGURES_DIR / "summary-metrics-bar-chart.png"
support_mix_path = FIGURES_DIR / "answer-rate-vs-supported-answer-rate.png"

scatter_df = comparison_df.dropna(subset=["overall_f1", "unsupported_answer_rate"]).copy()
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    scatter_df["unsupported_answer_rate"],
    scatter_df["overall_f1"],
    s=80,
    c=[colors.get(family, "#111827") for family in scatter_df["family"]],
)
_annotate_points(ax, scatter_df, x="unsupported_answer_rate", y="overall_f1")
ax.set_title("Overall F1 vs Unsupported-Answer Rate")
ax.set_xlabel("Unsupported-Answer Rate (%)")
ax.set_ylabel("Overall F1")
ax.axvline(20.0, color="#ef4444", linestyle="--", linewidth=1.2, label="20% budget")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(overall_vs_unsupported_path, dpi=200, bbox_inches="tight")
plt.close(fig)

scatter_df = comparison_df.dropna(subset=["answerable_f1", "unsupported_answer_rate"]).copy()
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    scatter_df["unsupported_answer_rate"],
    scatter_df["answerable_f1"],
    s=80,
    c=[colors.get(family, "#111827") for family in scatter_df["family"]],
)
_annotate_points(ax, scatter_df, x="unsupported_answer_rate", y="answerable_f1")
ax.set_title("Answerable F1 vs Unsupported-Answer Rate")
ax.set_xlabel("Unsupported-Answer Rate (%)")
ax.set_ylabel("Answerable F1")
ax.axvline(20.0, color="#ef4444", linestyle="--", linewidth=1.2, label="20% budget")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(answerable_vs_unsupported_path, dpi=200, bbox_inches="tight")
plt.close(fig)

bar_metrics = ["overall_f1", "answerable_f1", "unsupported_answer_rate", "abstain_f1"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, metric in zip(axes.flat, bar_metrics, strict=False):
    metric_df = comparison_df.dropna(subset=[metric]).copy()
    axis.bar(
        metric_df["label"].astype(str),
        metric_df[metric],
        color=[colors.get(family, "#111827") for family in metric_df["family"]],
    )
    axis.set_title(metric.replace("_", " ").title())
    axis.tick_params(axis="x", rotation=30)
    if metric == "unsupported_answer_rate":
        axis.axhline(20.0, color="#ef4444", linestyle="--", linewidth=1.2)
fig.tight_layout()
fig.savefig(summary_bar_path, dpi=200, bbox_inches="tight")
plt.close(fig)

support_df = comparison_df.dropna(subset=["answer_rate", "supported_answer_rate"]).copy()
if not support_df.empty:
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(
        support_df["answer_rate"],
        support_df["supported_answer_rate"],
        s=80,
        c=[colors.get(family, "#111827") for family in support_df["family"]],
    )
    _annotate_points(ax, support_df, x="answer_rate", y="supported_answer_rate")
    ax.set_title("Answer Rate vs Supported-Answer Rate")
    ax.set_xlabel("Answer Rate (%)")
    ax.set_ylabel("Supported-Answer Rate (%)")
    fig.tight_layout()
    fig.savefig(support_mix_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
else:
    support_mix_path = None
    print("Support-aware mix plot skipped because no loaded rows expose answer-rate and supported-answer-rate together.")

available_figure_paths = [
    overall_vs_unsupported_path,
    answerable_vs_unsupported_path,
    summary_bar_path,
]
if support_mix_path is not None:
    available_figure_paths.append(support_mix_path)

best_overall_row = comparison_df.loc[comparison_df["overall_f1"].astype(float).idxmax()].to_dict()
safest_row = comparison_df.loc[comparison_df["unsupported_answer_rate"].astype(float).idxmin()].to_dict()
constrained_df = comparison_df[
    comparison_df["unsupported_answer_rate"].notna()
    & (comparison_df["unsupported_answer_rate"] <= 20.0)
].copy()
best_under_budget_row = None
if not constrained_df.empty:
    best_under_budget_row = constrained_df.loc[constrained_df["overall_f1"].astype(float).idxmax()].to_dict()

comparison_summary = {
    "available_rows": [str(label) for label in comparison_df["label"].tolist()],
    "missing_rows": [spec["label"] for spec in missing_specs],
    "best_overall_f1_row": best_overall_row,
    "lowest_unsupported_answer_rate_row": safest_row,
    "best_overall_f1_under_20pct_budget": best_under_budget_row,
    "figure_paths": [str(path) for path in available_figure_paths],
}
COMPARISON_SUMMARY_JSON.write_text(json.dumps(comparison_summary, indent=2) + "\n", encoding="utf-8")

print("")
print(f"Saved comparison table: {COMPARISON_TABLE_CSV}")
print(f"Saved comparison records: {COMPARISON_TABLE_JSON}")
print(f"Saved comparison summary: {COMPARISON_SUMMARY_JSON}")
print("Saved figures:")
for figure_path in available_figure_paths:
    print(f"- {figure_path}")
print("")
print("Best overall F1 row:", best_overall_row["label"])
print("Lowest unsupported-answer rate row:", safest_row["label"])
if best_under_budget_row is None:
    print("No loaded row satisfies the 20% unsupported-answer budget.")
else:
    print("Best overall F1 row under the 20% unsupported-answer budget:", best_under_budget_row["label"])


## Comparison Note Template

Update the notes for the final analysis rather than a single training run.

Use this structure:

- Coverage
- Missing stages or missing artifacts
- Strongest result by overall F1
- Strongest result under the unsupported-answer constraint
- Best trade-off plot to cite in the paper
- Main takeaway
- Next paper edits


<h2 style="color: #15803d;">Save Notes And Review Comparison Artifacts</h2>

This cell saves the comparison table, figure list, and a note template for the final write-up. Use it after the comparison cell succeeds.


In [ ]:
captured_notebook_path = save_executed_notebook_snapshot()
mirrored_output_root = mirror_output_root(OUTPUT_ROOT)

if "comparison_df" not in globals():
    print("Run the comparison cell first so the metrics table and figure list exist.")
else:
    if not RUN_NOTES_FILE.exists():
        available_rows = comparison_df["label"].astype(str).tolist()
        figure_paths = []
        if COMPARISON_SUMMARY_JSON.exists():
            summary_payload = json.loads(COMPARISON_SUMMARY_JSON.read_text(encoding="utf-8"))
            figure_paths = summary_payload.get("figure_paths", [])

        RUN_NOTES_FILE.write_text(
            "\n".join(
                [
                    "# KeelNet Final Comparison Notes",
                    "",
                    "## Coverage",
                    *[f"- {label}" for label in available_rows],
                    "",
                    "## Missing Stages Or Missing Artifacts",
                    *([f"- {label}" for label in MISSING_STAGE_LABELS] if MISSING_STAGE_LABELS else ["- none"]),
                    "",
                    "## Output Paths",
                    f"- Output folder: {OUTPUT_ROOT}",
                    f"- Figures dir: {FIGURES_DIR}",
                    f"- Comparison CSV: {COMPARISON_TABLE_CSV}",
                    f"- Comparison JSON: {COMPARISON_TABLE_JSON}",
                    f"- Comparison summary JSON: {COMPARISON_SUMMARY_JSON}",
                    f"- Executed notebook archive target: {EXECUTED_NOTEBOOK_PATH}",
                    "",
                    "## Figures To Review",
                    *([f"- {path}" for path in figure_paths] if figure_paths else ["- run the comparison cell first"]),
                    "",
                    "## Main Takeaway",
                    "- ",
                    "",
                    "## Paper Follow-Ups",
                    "1. ",
                    "2. ",
                    "3. ",
                ]
            )
            + "\n",
            encoding="utf-8",
        )

    RUN_SUMMARY_FILE.write_text(
        json.dumps(
            {
                "notebook": NOTEBOOK_TITLE,
                "run_name": RUN_NAME,
                "runtime_mode": RUNTIME_MODE,
                "git_branch": CURRENT_BRANCH,
                "output_root": str(OUTPUT_ROOT),
                "figures_dir": str(FIGURES_DIR),
                "comparison_csv": str(COMPARISON_TABLE_CSV),
                "comparison_json": str(COMPARISON_TABLE_JSON),
                "comparison_summary_json": str(COMPARISON_SUMMARY_JSON),
                "available_rows": comparison_df["label"].astype(str).tolist(),
                "missing_rows": MISSING_STAGE_LABELS,
                "executed_notebook_dir": str(NOTEBOOK_ARCHIVE_DIR),
                "executed_notebook_target": str(EXECUTED_NOTEBOOK_PATH),
                "executed_notebook_saved": captured_notebook_path is not None,
                "executed_notebook_instructions_file": str(EXECUTED_NOTEBOOK_INSTRUCTIONS_FILE),
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

print(f"Notes template: {RUN_NOTES_FILE}")
print(f"Run summary: {RUN_SUMMARY_FILE}")
print(f"Executed notebook target: {EXECUTED_NOTEBOOK_PATH}")
if captured_notebook_path is None:
    print(
        "Automatic notebook capture was unavailable. "
        f"If you want the notebook archive, save it manually to {EXECUTED_NOTEBOOK_PATH}."
    )
if mirrored_output_root is not None:
    print(f"Drive mirror: {mirrored_output_root}")
print("Current files under OUTPUT_ROOT:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    print(path)


<h2 style="color: #15803d;">Final Check</h2>

A useful wrap-up comparison is not just a pretty graph.

Check all four:

- the rows come from real saved artifacts, not manual copying
- the stage labels match the eval files they came from
- the graphs reflect the same metrics the paper discusses
- the takeaway is honest about missing stages or failed runs


<h2 style="color: #15803d;">Share This Comparison</h2>

This cell saves a small share note and marks the comparison run as completed. Use it after you review the figures and decide which ones belong in the paper or slides.


In [ ]:
from datetime import datetime, timezone

captured_notebook_path = save_executed_notebook_snapshot()
mirrored_output_root = mirror_output_root(OUTPUT_ROOT)

available_rows = comparison_df["label"].astype(str).tolist() if "comparison_df" in globals() else []
best_overall_label = "<run comparison cell first>"
lowest_unsupported_label = "<run comparison cell first>"
figure_paths = []
if COMPARISON_SUMMARY_JSON.exists():
    summary_payload = json.loads(COMPARISON_SUMMARY_JSON.read_text(encoding="utf-8"))
    best_overall = summary_payload.get("best_overall_f1_row")
    safest = summary_payload.get("lowest_unsupported_answer_rate_row")
    if isinstance(best_overall, dict):
        best_overall_label = str(best_overall.get("label"))
    if isinstance(safest, dict):
        lowest_unsupported_label = str(safest.get("label"))
    figure_paths = summary_payload.get("figure_paths", [])

share_lines = [
    "# KeelNet Final Comparison Share Note",
    "",
    f"- runtime mode: {RUNTIME_MODE}",
    f"- branch name: {CURRENT_BRANCH}",
    f"- RUN_NAME: {RUN_NAME}",
    f"- available rows: {', '.join(available_rows) if available_rows else 'run comparison cell first'}",
    f"- best overall F1 row: {best_overall_label}",
    f"- lowest unsupported-answer-rate row: {lowest_unsupported_label}",
    f"- comparison CSV: {COMPARISON_TABLE_CSV}",
    f"- comparison summary JSON: {COMPARISON_SUMMARY_JSON}",
    f"- executed notebook archive target: {EXECUTED_NOTEBOOK_PATH}",
    "",
    "## Figure Paths",
    *([f"- {path}" for path in figure_paths] if figure_paths else ["- run comparison cell first"]),
]
if mirrored_output_root is not None:
    share_lines.append(f"- Drive mirror path: {mirrored_output_root}")

share_note = "\n".join(share_lines) + "\n"
SHARE_NOTE_FILE = OUTPUT_ROOT / "collab-share-note.md"
SHARE_NOTE_FILE.write_text(share_note, encoding="utf-8")
COMPLETION_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"completed_at={datetime.now(timezone.utc).isoformat()}",
            f"share_note={SHARE_NOTE_FILE.name}",
            "status=completed",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(share_note)
if captured_notebook_path is None:
    print(
        "Automatic notebook capture was unavailable. "
        f"If you want the notebook archive, save it manually to {EXECUTED_NOTEBOOK_PATH}."
    )
print(f"Saved share note: {SHARE_NOTE_FILE}")
print(f"Saved completion marker: {COMPLETION_MARKER_FILE}")
if mirrored_output_root is not None:
    mirror_output_root(OUTPUT_ROOT)
